# 🧩 Mini-Lab: Validate at the Boundaries, Configure from Env

**Module 7: Trust but Verify** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Define** input and output schemas and reject seeded bad payloads
2. **Load** a `Settings` object from env vars + `.env`
3. **Confirm** `.env` is git-ignored and `.env.example` has no secrets

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Boundary validation | Check user input and model output — the parts you don't control |
| Layered configuration | Defaults in code, overrides in environment, secrets never in Git |


## Why Boundaries?

In an AI pipeline you typically control the code in the middle, but **not** the data at the edges:

| Boundary | Example of bad data |
|----------|---------------------|
| **User input** (HTTP request, form, CLI) | Missing field, wrong type, empty string |
| **Model output** (LLM JSON, API response) | Extra keys, null where you expect a list, hallucinated schema |

If you validate **at the boundary**, the rest of your code can trust the data it receives.

## Step 1 — Define a Schema with `@dataclass` and Type Hints

We'll model a simple extraction request: a user sends text and asks the LLM to pull out entities.

In [1]:
from dataclasses import dataclass, field


@dataclass
class ExtractionRequest:
    """Schema for an incoming extraction request."""
    text: str
    entity_types: list[str]
    max_entities: int = 10  # sensible default


# A well-formed request
good = ExtractionRequest(
    text="OpenAI was founded in 2015 in San Francisco.",
    entity_types=["person", "organization", "location"],
)
print(good)

ExtractionRequest(text='OpenAI was founded in 2015 in San Francisco.', entity_types=['person', 'organization', 'location'], max_entities=10)


The dataclass gives us a clear **shape**: we know exactly which fields exist and what types they should be. But a dataclass alone won't reject bad data — we need a validation step.

## Step 2 — Write a Boundary Validator

We write a small `validate_request` function that checks the raw dictionary (as it would arrive from an HTTP body or JSON file) and either returns a clean `ExtractionRequest` or raises a clear error.

In [2]:
class ValidationError(Exception):
    """Raised when input data fails boundary validation."""


def validate_request(raw: dict) -> ExtractionRequest:
    """Validate a raw dict at the system boundary and return a typed object."""
    errors: list[str] = []

    # --- Check required fields exist ---
    if "text" not in raw:
        errors.append("Missing required field: 'text'")
    elif not isinstance(raw["text"], str) or not raw["text"].strip():
        errors.append("'text' must be a non-empty string")

    if "entity_types" not in raw:
        errors.append("Missing required field: 'entity_types'")
    elif not isinstance(raw["entity_types"], list) or len(raw["entity_types"]) == 0:
        errors.append("'entity_types' must be a non-empty list")
    elif not all(isinstance(t, str) for t in raw["entity_types"]):
        errors.append("Every item in 'entity_types' must be a string")

    # --- Optional field with type check ---
    max_entities = raw.get("max_entities", 10)
    if not isinstance(max_entities, int) or max_entities < 1:
        errors.append("'max_entities' must be a positive integer")

    # --- Fail loudly with ALL errors at once ---
    if errors:
        raise ValidationError("Input validation failed:\n  • " + "\n  • ".join(errors))

    return ExtractionRequest(
        text=raw["text"].strip(),
        entity_types=raw["entity_types"],
        max_entities=max_entities,
    )

## Step 3 — Test with Good and Bad Payloads

Let's throw six payloads at the validator — three good, three bad — and see what happens.

In [3]:
test_payloads = [
    # ✅ Valid
    {"text": "Tesla reported $25B revenue.", "entity_types": ["organization", "money"]},
    # ✅ Valid with optional override
    {"text": "Alice met Bob in Paris.", "entity_types": ["person"], "max_entities": 5},
    # ❌ Missing 'text'
    {"entity_types": ["person"]},
    # ❌ Empty entity_types list
    {"text": "Some text here.", "entity_types": []},
    # ❌ Wrong type for text + bad max_entities
    {"text": 12345, "entity_types": ["org"], "max_entities": -1},
    # ❌ entity_types contains non-strings
    {"text": "Hello world.", "entity_types": ["person", 99]},
]

for i, payload in enumerate(test_payloads, 1):
    try:
        result = validate_request(payload)
        print(f"Payload {i} ✅  → {result}")
    except ValidationError as exc:
        print(f"Payload {i} ❌  → {exc}")
    print()

Payload 1 ✅  → ExtractionRequest(text='Tesla reported $25B revenue.', entity_types=['organization', 'money'], max_entities=10)

Payload 2 ✅  → ExtractionRequest(text='Alice met Bob in Paris.', entity_types=['person'], max_entities=5)

Payload 3 ❌  → Input validation failed:
  • Missing required field: 'text'

Payload 4 ❌  → Input validation failed:
  • 'entity_types' must be a non-empty list

Payload 5 ❌  → Input validation failed:
  • 'text' must be a non-empty string
  • 'max_entities' must be a positive integer

Payload 6 ❌  → Input validation failed:
  • Every item in 'entity_types' must be a string



Notice that **Payload 5** reports *two* errors at once — both the bad `text` type and the negative `max_entities`. Collecting all errors before raising gives the caller a single, complete error message instead of a frustrating fix-one-resubmit loop.

## Step 4 — Validate the Output Boundary

Model outputs are just as untrustworthy as user inputs. Let's define a schema for what we expect an LLM extraction to return, then validate simulated model output.

In [4]:
import json


@dataclass
class ExtractedEntity:
    """One entity extracted by the model."""
    name: str
    entity_type: str
    confidence: float


@dataclass
class ExtractionResponse:
    """Schema for the model's extraction output."""
    entities: list[ExtractedEntity]


def validate_response(raw_json: str) -> ExtractionResponse:
    """Validate model output at the output boundary."""
    # --- Parse JSON ---
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError as exc:
        raise ValidationError(f"Model returned invalid JSON: {exc}")

    if "entities" not in data or not isinstance(data["entities"], list):
        raise ValidationError("Model output missing 'entities' list")

    validated: list[ExtractedEntity] = []
    for idx, item in enumerate(data["entities"]):
        errors: list[str] = []
        if not isinstance(item.get("name"), str) or not item["name"].strip():
            errors.append(f"Entity [{idx}]: 'name' must be a non-empty string")
        if not isinstance(item.get("entity_type"), str):
            errors.append(f"Entity [{idx}]: 'entity_type' must be a string")
        conf = item.get("confidence")
        if not isinstance(conf, (int, float)) or not (0.0 <= conf <= 1.0):
            errors.append(f"Entity [{idx}]: 'confidence' must be a float between 0 and 1")
        if errors:
            raise ValidationError("Output validation failed:\n  • " + "\n  • ".join(errors))
        validated.append(ExtractedEntity(
            name=item["name"].strip(),
            entity_type=item["entity_type"],
            confidence=float(conf),
        ))

    return ExtractionResponse(entities=validated)

In [5]:
# Simulated model outputs — one good, two bad
model_outputs = [
    # ✅ Well-formed
    '{"entities": [{"name": "OpenAI", "entity_type": "organization", "confidence": 0.95}]}',
    # ❌ Invalid JSON (trailing comma)
    '{"entities": [{"name": "OpenAI",}]}',
    # ❌ confidence out of range
    '{"entities": [{"name": "Paris", "entity_type": "location", "confidence": 1.5}]}',
]

for i, output in enumerate(model_outputs, 1):
    try:
        result = validate_response(output)
        print(f"Output {i} ✅  → {result}")
    except ValidationError as exc:
        print(f"Output {i} ❌  → {exc}")
    print()

Output 1 ✅  → ExtractionResponse(entities=[ExtractedEntity(name='OpenAI', entity_type='organization', confidence=0.95)])

Output 2 ❌  → Model returned invalid JSON: Expecting property name enclosed in double quotes: line 1 column 33 (char 32)

Output 3 ❌  → Output validation failed:
  • Entity [0]: 'confidence' must be a float between 0 and 1



---
## Settings from the environment

Same sitting: validate IO, then load config from env.


## Why Layered Configuration?

Hard-coding settings in source code is fragile and dangerous:

| Problem | Example |
|---------|---------|
| **Leaked secrets** | `api_key = "sk-abc123"` pushed to GitHub |
| **Inflexible deployments** | Changing a timeout means editing code and redeploying |
| **No per-environment control** | Dev and prod use the same values |

The **layered configuration** pattern solves all three. Settings come from three sources, each overriding the last:

```
Layer 1 (lowest priority):  Hard-coded DEFAULTS in code
Layer 2:                    .env file  (local dev, git-ignored)
Layer 3 (highest priority): Real ENVIRONMENT VARIABLES  (CI/CD, containers)
```

This means you always have a working default, developers can customize via `.env`, and production overrides via real environment variables win.

## Step 1 — Install `python-dotenv`

`python-dotenv` reads key-value pairs from a `.env` file into environment variables. It's the standard lightweight tool for this pattern.

In [1]:
# !pip install -q python-dotenv

## Step 2 — Create a `.env` File

In a real project you'd create `.env` by hand in your project root. Here we'll write one programmatically so the lab is self-contained.

> **Key rule:** This file is for *local development only*. It must **never** be committed to version control.

In [2]:
from pathlib import Path

env_content = """# Local development settings - DO NOT COMMIT
OPENAI_API_KEY=sk-fake-dev-key-12345
MODEL_NAME=gpt-4o-mini
REQUEST_TIMEOUT=45
"""

Path(".env").write_text(env_content.strip(), encoding="utf-8")
print("Created .env file:")
print(Path(".env").read_text(encoding="utf-8"))

Created .env file:
# Local development settings - DO NOT COMMIT
OPENAI_API_KEY=sk-fake-dev-key-12345
MODEL_NAME=gpt-4o-mini
REQUEST_TIMEOUT=45


## Step 3 — Add `.env` to `.gitignore`

Before doing anything else, ensure `.env` is in your `.gitignore`. If the file doesn't exist, we create one. If it does, we append the rule only if it's missing.

> **Why before anything else?** The moment `.env` exists on disk, a careless `git add .` could commit it. Adding the ignore rule *first* prevents that.

In [3]:
gitignore_path = Path(".gitignore")

# Read existing content (or start empty)
existing = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""

# Only append if not already present
if ".env" not in existing.splitlines():
    with open(gitignore_path, "a", encoding="utf-8") as f:
        f.write("\n.env\n")
    print("✅ Added .env to .gitignore")
else:
    print("✅ .env already in .gitignore")

✅ .env already in .gitignore


## Step 4 — Build the Layered Settings Object

We define a `Settings` dataclass with sensible defaults (Layer 1), then load values from the `.env` file (Layer 2), and finally allow real environment variables to override everything (Layer 3).

The function `load_settings()` implements all three layers in order.

In [4]:
import os
from dataclasses import dataclass
from dotenv import load_dotenv


@dataclass
class Settings:
    """Application settings built from layered configuration."""
    openai_api_key: str
    model_name: str
    request_timeout: int
    max_retries: int


# --- Layer 1: Hard-coded defaults ---
DEFAULTS = {
    "OPENAI_API_KEY": "",          # no default — must be provided
    "MODEL_NAME": "gpt-4o-mini",
    "REQUEST_TIMEOUT": "30",        # env vars are always strings
    "MAX_RETRIES": "3",
}


def load_settings(env_path: str = ".env") -> Settings:
    """Load settings: defaults → .env file → environment variables."""

    # --- Layer 2: .env file (does NOT override already-set env vars) ---
    load_dotenv(env_path, override=False, encoding="utf-8")

    # --- Layer 3: os.environ (real env vars) always wins ---
    # We read from os.environ, falling back to our defaults.
    def get(key: str) -> str:
        return os.environ.get(key, DEFAULTS[key])

    api_key = get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY is required but not set in .env or environment")

    return Settings(
        openai_api_key=api_key,
        model_name=get("MODEL_NAME"),
        request_timeout=int(get("REQUEST_TIMEOUT")),
        max_retries=int(get("MAX_RETRIES")),
    )

## Step 5 — See the Layers in Action

Let's load settings and inspect which values came from which layer.

In [5]:
settings = load_settings()
print(settings)

Settings(openai_api_key='sk-fake-dev-key-12345', model_name='gpt-4o-mini', request_timeout=45, max_retries=3)


Let's trace where each value came from:

| Setting | Value | Source |
|---------|-------|--------|
| `openai_api_key` | `sk-fake-dev-key-12345` | **Layer 2** — `.env` file |
| `model_name` | `gpt-4o-mini` | **Layer 2** — `.env` (same as default, but set in `.env`) |
| `request_timeout` | `45` | **Layer 2** — `.env` overrode the default of `30` |
| `max_retries` | `3` | **Layer 1** — default (not in `.env`, not in environment) |

## Step 6 — Layer 3 Overrides Everything

Real environment variables (Layer 3) take the highest priority. Let's simulate setting one and see it override the `.env` value.

In [6]:
# Simulate a real env var (as CI/CD or Docker would set it)
os.environ["REQUEST_TIMEOUT"] = "120"

settings = load_settings()
print(f"request_timeout = {settings.request_timeout}")  # 120, not 45
print("↑ Real env var (Layer 3) overrode .env (Layer 2)")

# Clean up so it doesn't affect later cells
del os.environ["REQUEST_TIMEOUT"]

request_timeout = 120
↑ Real env var (Layer 3) overrode .env (Layer 2)


## Step 7 — Secrets Hygiene: What NOT to Do

Let's see the **wrong** way and the **right** way side by side.

In [7]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
### ❌ Never do this

```python
# Hard-coded in source code — gets committed to git!
api_key = "sk-real-secret-key-abc123"
```

```python
# Printing or logging secrets
print(f"Using key: {settings.openai_api_key}")
logger.info(f"API key = {api_key}")
```

### ✅ Do this instead

```python
# Load from environment
settings = load_settings()

# Log only a safe suffix
print(f"API key loaded: ...{settings.openai_api_key[-4:]}")
```

### ✅ Always git-ignore your `.env`

```
# .gitignore
.env
```
""")


### ❌ Never do this

```python
# Hard-coded in source code — gets committed to git!
api_key = "sk-real-secret-key-abc123"
```

```python
# Printing or logging secrets
print(f"Using key: {settings.openai_api_key}")
logger.info(f"API key = {api_key}")
```

### ✅ Do this instead

```python
# Load from environment
settings = load_settings()

# Log only a safe suffix
print(f"API key loaded: ...{settings.openai_api_key[-4:]}")
```

### ✅ Always git-ignore your `.env`

```
# .gitignore
.env
```


## Step 8 — Fail Loudly on Missing Secrets

Our `load_settings()` already raises a `ValueError` if the API key is empty. Let's verify that it works — a missing secret should fail at startup, not silently halfway through a pipeline.

In [8]:
# Write a .env without the API key
Path(".env").write_text("MODEL_NAME=gpt-4o-mini\n", encoding="utf-8")

# Remove from environment too, so no layer provides it
os.environ.pop("OPENAI_API_KEY", None)

try:
    bad_settings = load_settings()
except ValueError as exc:
    print(f"✅ Caught expected error: {exc}")

# Restore the original .env for any re-runs
Path(".env").write_text(env_content.strip(), encoding="utf-8")

✅ Caught expected error: OPENAI_API_KEY is required but not set in .env or environment


123

## Cleanup

In [9]:
# Remove the demo files so they don't linger
Path(".env").unlink(missing_ok=True)
print("Removed .env demo file")

Removed .env demo file


## 🎯 Summary

### Key Takeaways

1. **Boundary validation** — check data where it enters (user input) and where it leaves (model output) your system, so inner code can trust it
2. **Dataclass schemas** — `@dataclass` declares the exact shape (fields and defaults) you expect, serving as living documentation
3. **Type hints** — annotating fields with `str`, `int`, `list[str]`, `float` makes expectations explicit and enables runtime checks
4. **Collect all errors** — reporting every validation failure at once gives callers a single, actionable error message instead of a fix-one-resubmit loop
5. **Fail loudly** — a clear `ValidationError` at the boundary is far easier to debug than a mysterious `KeyError` deep inside your pipeline